In [5]:
import sys
import os
import json
import torch
from torch.utils.data import Dataset, DataLoader
from PIL import Image

# Ensure project structure is accessible
sys.path.append("../")

from data.text_tokenizer import CLEVRTokenizer
from configs.data_config import get_config
from data.transforms import CLIPTransforms, DINOMultiCropTransforms, LinearProbeTransforms
from data.clevr_dataset import CLEVRCollateFn

# =========================================================================
# 1. CHOKE POINT PATCHER (Safe Subclass Bypass)
# =========================================================================
class PatchedCLEVRDataset(Dataset):
    def __init__(self, config, mode: str, split: str = "train", transform=None, tokenizer=None):
        self.mode = mode
        self.split = split
        self.transform = transform
        self.tokenizer = tokenizer
        
        # [CHOKE POINT 1: Hierarchy Patching]
        if mode in ["clip", "dino"]:
            base_dir = config.base_dir_part_a
            self.image_dir = os.path.join(base_dir, split, "images")
            json_path = os.path.join(base_dir, split, f"clevr_{split}_captions.json")
            
            with open(json_path, 'r') as f:
                self.annotations = json.load(f)
                
            # Flatten wrapper if it exists (Safeguard)
            if isinstance(self.annotations, dict) and "examples" in self.annotations:
                self.annotations = self.annotations["examples"]
                
        else:
            base_dir = config.base_dir_part_aa
            self.image_dir = os.path.join(base_dir, "Clevr_official", "images", split)
            
            count_json = os.path.join(base_dir, "Probe-Datasets", f"clevr_count_{split}.json")
            color_json = os.path.join(base_dir, "Probe-Datasets", f"clevr_colors_{split}.json")
            
            with open(count_json, 'r') as fc, open(color_json, 'r') as fp:
                # [CHOKE POINT 3: Extract from "examples" dict wrapper]
                count_examples = json.load(fc)["examples"]
                color_examples = json.load(fp)["examples"]
                
            self.counts = []
            self.color_sets = []
            self.annotations = []
            
            # [CHOKE POINT 4: Dual-JSON Zipping]
            for count_ex, color_ex in zip(count_examples, color_examples):
                # [CHOKE POINT 5: Bypass string error and capture "multi_hot" natively]
                self.counts.append(count_ex["label"])
                self.color_sets.append(torch.tensor(color_ex["multi_hot"], dtype=torch.float32))
                self.annotations.append({"image_filename": count_ex["image_filename"]})

    def __len__(self):
        return len(self.annotations)

    def __getitem__(self, idx):
        ann = self.annotations[idx]
        
        # [CHOKE POINT 2: Image Key Dict Fix] ("image_filename" over "image")
        img_filename = ann.get("image_filename", ann.get("image", ann.get("filename", "")))
        img_path = os.path.join(self.image_dir, img_filename)
        
        image = Image.open(img_path).convert("RGB")

        if self.mode == "clip":
            img_tensor = self.transform(image)
            caption = ann.get("caption", "")
            tokens, mask = self.tokenizer.encode(caption)
            return {
                "image": img_tensor,
                "tokens": tokens,
                "padding_mask": mask,
                "raw_caption": caption
            }
        elif self.mode == "dino":
            return self.transform(image)
        elif self.mode == "linear_probe":
            return {
                "image": self.transform(image),
                "count_label": self.counts[idx],
                "color_label": self.color_sets[idx]
            }



['/Users/rishit/miniconda3/envs/COL775/lib/python310.zip',
 '/Users/rishit/miniconda3/envs/COL775/lib/python3.10',
 '/Users/rishit/miniconda3/envs/COL775/lib/python3.10/lib-dynload',
 '',
 '/Users/rishit/miniconda3/envs/COL775/lib/python3.10/site-packages',
 '/Users/rishit/Documents/Programming/Courses/COL775/Assignment-2_COL775',
 '/var/folders/ql/_bzwm91d65gb17mvkg17ksb40000gn/T/tmp7a9ejs22',
 '../',
 '../']

In [10]:

# =========================================================================
# 2. TRIAL RUN SUITE
# =========================================================================
def run_dataloader_trials():
    print("Setting up Config & Tokenizer...")
    
    cfg = get_config("local")
    cfg.base_dir_part_a = "../../data/A2_dataset/Part_A"
    cfg.base_dir_part_aa = "../../data/A2_dataset/Part_Aa"
    cfg.batch_size = 2 # Using a small chunk for the trial run
    
    # Initialize task-specific Tokenizer from captions directly
    tokenizer = CLEVRTokenizer(max_seq_len=77)
    train_caption_json = os.path.join(cfg.base_dir_part_a, "train", "clevr_train_captions.json")
    tokenizer.build_vocab(train_caption_json)
    
    print(f"Tokenizer Built. Total Vocab Size: {tokenizer.vocab_size}\n")
    
    trials = [
        ("clip",         CLIPTransforms(224)),
        ("dino",         DINOMultiCropTransforms(8)),
        ("linear_probe", LinearProbeTransforms(224))
    ]
    
    # Iterate through each dataset logic structure mapping
    for mode, transform in trials:
        print(f"[{mode.upper()} DATALOADER] Initialization...")
        dataset = PatchedCLEVRDataset(cfg, mode=mode, split="train", transform=transform, tokenizer=tokenizer)
        collate_fn = CLEVRCollateFn(mode=mode)
        
        dl = DataLoader(dataset, batch_size=cfg.batch_size, collate_fn=collate_fn, shuffle=False)
        
        batch = next(iter(dl))
        
        print(f" > Yielded 1st batch for {mode} gracefully!")
        if mode == "clip":
            print(f"   Images Shape: {batch['images'].shape}")
            print(f"   Tokens Shape: {batch['tokens'].shape}")
            raw_caption_preview = [caption[:35] + '...' if len(caption) > 35 else caption for caption in batch['raw_captions']]
            print(f"   Raw Captions: {raw_caption_preview}\n")
            
        elif mode == "dino":
            print(f"   Global Crops Shape (Explicit 5D Expected): {batch['global_crops'].shape}")
            print(f"   Local Crops Shape (Explicit 5D Expected): {batch['local_crops'].shape}\n")
            
        elif mode == "linear_probe":
            print(f"   Images Shape: {batch['images'].shape}")
            print(f"   Count Labels Tensor: {batch['count_labels']} | Object Count Types: {batch['count_labels'].dtype}")
            print(f"   Color Sets Multi-hot: {batch['color_labels'].shape} | First Matrix: {batch['color_labels'][0]}\n")

if __name__ == "__main__":
    run_dataloader_trials()

Setting up Config & Tokenizer...
Tokenizer Built. Total Vocab Size: 36

[CLIP DATALOADER] Initialization...
 > Yielded 1st batch for clip gracefully!
   Images Shape: torch.Size([2, 3, 224, 224])
   Tokens Shape: torch.Size([2, 77])
   Raw Captions: ['An image with 5 objects: 1 large gr...', 'An image with 7 objects: 1 large gr...']

[DINO DATALOADER] Initialization...
 > Yielded 1st batch for dino gracefully!
   Global Crops Shape (Explicit 5D Expected): torch.Size([2, 2, 3, 224, 224])
   Local Crops Shape (Explicit 5D Expected): torch.Size([2, 8, 3, 96, 96])

[LINEAR_PROBE DATALOADER] Initialization...
 > Yielded 1st batch for linear_probe gracefully!
   Images Shape: torch.Size([2, 3, 224, 224])
   Count Labels Tensor: tensor([6, 9]) | Object Count Types: torch.int64
   Color Sets Multi-hot: torch.Size([2, 8]) | First Matrix: tensor([1., 0., 1., 1., 1., 0., 1., 0.])

